# Pelajaran 13 - Memori Agen dengan Graf Pengetahuan Cognee


## Setup

Notebook ini menunjukkan cara membina **pembantu pengekodan** yang pintar dengan memori berterusan menggunakan graf pengetahuan [**Cognee**](https://www.cognee.ai/) dan **Microsoft Agent Framework** (MAF).

Cognee menukar teks tidak berstruktur kepada graf pengetahuan berstruktur yang boleh disoal, disokong oleh penyerapan vektor — memberikan ejen anda memori jangka panjang yang kaya dan sedar hubungan.

### Apa Yang Akan Anda Pelajari
1. **Membina Graf Pengetahuan**: Menukar profil pembangun dan amalan terbaik kepada pengetahuan berstruktur yang boleh disoal.
2. **Mengintegrasi Cognee dengan MAF**: Menggunakan fungsi `@tool` untuk membolehkan ejen MAF menyoal graf pengetahuan Cognee.
3. **Perbualan Berinformasi Sesi**: Mengekalkan konteks merentasi pelbagai soalan dalam sesi yang sama.
4. **Memori Jangka Panjang**: Menyimpan pengetahuan penting merentasi sesi dan mendapatkannya kembali dalam perbualan baru.

### Prasyarat
- Python 3.9+
- Redis dijalankan secara tempatan (`docker run -d -p 6379:6379 redis`) untuk pengurusan sesi
- Kunci API LLM (contoh OpenAI) — tetapkan `LLM_API_KEY` dalam `.env`
- `CACHING=true` dalam `.env` (diperlukan untuk sesi Cognee)
- Projek Azure AI Foundry dengan model sembang yang dipasang
- `AZURE_AI_PROJECT_ENDPOINT` dan `AZURE_AI_MODEL_DEPLOYMENT_NAME` dalam `.env`
- CLI Azure diautentikasi (`az login`)


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity "cognee[redis]==0.4.0" -q

In [ ]:
import os
from pathlib import Path
from typing import Annotated

from dotenv import load_dotenv

load_dotenv()

os.environ["LLM_API_KEY"] = os.getenv("LLM_API_KEY", "")
os.environ["CACHING"] = os.getenv("CACHING", "true")

import cognee
from cognee.modules.search.types import SearchType

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

print(f"Cognee version: {cognee.__version__}")
print(f"CACHING: {os.environ.get('CACHING')}")

In [ ]:
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

print("✅ AzureAIProjectAgentProvider created")

## Jenis Memori Ejen

Notebook ini meneroka tiga jenis memori yang sama dari notebook Utama Pelajaran 13, tetapi menggunakan Cognee sebagai backend memori jangka panjang:

| Jenis Memori | Mekanisme | Tempoh Masa |
|---|---|---|
| **Kerja** | `agent.create_session()` (MAF) | Perbualan tunggal |
| **Jangka pendek** | Cache sesi Cognee (Redis) | Sesi tunggal |
| **Jangka panjang** | Graf pengetahuan Cognee + vektor | Kekal |

### Seni Bina Memori Cognee
```
┌──────────────────────────┐
│      Raw Data            │  (developer profiles, docs, conversations)
└───────────┬──────────────┘
            │  cognee.add() + cognee.cognify()
            ▼
┌──────────────────────────────────────────┐
│  Knowledge Graph + Vector Embeddings     │
└───────────┬──────────────────────────────┘
            │  cognee.search()
            ▼
┌──────────────────┐       ┌────────────────┐
│  MAF Agent       │──────▶│  @tool funcs   │
│  (AgentSession)  │       │  wrapping       │
│                  │       │  cognee.search  │
└──────────────────┘       └────────────────┘
```


## Sediakan Simpanan Cognee


In [ ]:
DATA_ROOT = Path('.data_storage').resolve()
SYSTEM_ROOT = Path('.cognee_system').resolve()

DATA_ROOT.mkdir(parents=True, exist_ok=True)
SYSTEM_ROOT.mkdir(parents=True, exist_ok=True)

cognee.config.data_root_directory(str(DATA_ROOT))
cognee.config.system_root_directory(str(SYSTEM_ROOT))

await cognee.prune.prune_data()
await cognee.prune.prune_system(metadata=True)
print("✅ Cognee storage configured and reset")

## Bahagian 1 — Membina Pangkalan Pengetahuan

Kami memasukkan tiga jenis data untuk mencipta pangkalan pengetahuan yang menyeluruh untuk pembantu pengekodan kami:

1. **Profil Pembangun** — kepakaran peribadi dan latar belakang teknikal
2. **Amalan Terbaik Python** — Zen Python dengan panduan praktikal
3. **Perbualan Sejarah** — sesi soal jawab lalu antara pembangun dan pembantu AI


In [ ]:
developer_intro = (
    "Hi, I'm an AI/Backend engineer. "
    "I build FastAPI services with Pydantic, heavy asyncio/aiohttp pipelines, "
    "and production testing via pytest-asyncio. "
    "I've shipped low-latency APIs on AWS, Azure, and GoogleCloud."
)

python_zen_principles = """
# The Zen of Python: Practical Guide

## Key Principles With Guidance

### 1. Beautiful is better than ugly
Prefer descriptive names, clear structure, and consistent formatting.

### 2. Explicit is better than implicit
Be clear about behavior, imports, and types.

### 3. Simple is better than complex
Choose straightforward solutions first.

### 4. Flat is better than nested
Use early returns to reduce indentation.

## Modern Python Tie-ins
- Type hints reinforce explicitness
- Context managers enforce safe resource handling
- Dataclasses improve readability for data containers
"""

human_agent_conversations = """
"conversations": [
    {
      "topic": "async/await patterns",
      "user_query": "I'm building a web scraper that needs to handle thousands of URLs concurrently. What's the best way to structure this with asyncio?",
      "assistant_response": "Use asyncio with aiohttp, a semaphore to cap concurrency, TCPConnector for connection pooling, and context managers for session lifecycle."
    },
    {
      "topic": "dataclass vs pydantic",
      "user_query": "When should I use dataclasses vs Pydantic models?",
      "assistant_response": "For API input/output, prefer Pydantic: runtime validation, type coercion, JSON serialization. Integrates cleanly with FastAPI."
    },
    {
      "topic": "testing patterns",
      "user_query": "What's the best approach for pytest with async functions?",
      "assistant_response": "Use pytest-asyncio, async fixtures, and an isolated test database or mocks to reliably test async code."
    },
    {
      "topic": "error handling and logging",
      "user_query": "What's the best approach for production-ready error management?",
      "assistant_response": "Centralized error handling with custom exceptions, structured logging, and FastAPI middleware."
    }
  ]
"""

print("✅ Data sources prepared")

In [ ]:
await cognee.add(developer_intro, node_set=["developer_data"])
await cognee.add(human_agent_conversations, node_set=["developer_data"])
await cognee.add(python_zen_principles, node_set=["principles_data"])

await cognee.cognify()
print("✅ Knowledge graph built")

## Visualisasikan Graf Pengetahuan

Cognee dapat memaparkan visualisasi HTML interaktif bagi entiti dan hubungan yang telah diekstraknya.


In [ ]:
from cognee import visualize_graph

await visualize_graph('./cognee_graph.html')
print("📊 Graph saved to cognee_graph.html — open it in a browser to explore.")

## Memperkaya Memori dengan Memify

`memify()` menganalisis graf pengetahuan dan menghasilkan peraturan pintar — mengenal pasti corak, amalan terbaik, dan hubungan antara konsep.


In [ ]:
await cognee.memify()
print("✅ Memory enriched with memify")

## Part 2 — Ejen MAF dengan Alat Cognee

Kini kita mencipta ejen MAF yang boleh memanggil graf pengetahuan Cognee melalui fungsi `@tool`. Ini membolehkan ejen memanfaatkan sepenuhnya kuasa carian semantik yang menyedari graf sambil mengekalkan konteks perbualan melalui sesi.


In [ ]:
@tool(approval_mode="never_require")
async def search_knowledge(
    query: Annotated[str, "Natural-language question to search the knowledge graph"],
) -> str:
    """Search the Cognee knowledge graph for relevant developer knowledge, best practices, and past conversations."""
    results = await cognee.search(
        query_text=query,
        query_type=SearchType.GRAPH_COMPLETION,
    )
    if not results:
        return "No relevant knowledge found."
    return str(results)


@tool(approval_mode="never_require")
async def search_principles(
    query: Annotated[str, "Question about Python principles or best practices"],
) -> str:
    """Search only the Python principles subset of the knowledge graph."""
    from cognee.modules.engine.models.node_set import NodeSet
    results = await cognee.search(
        query_text=query,
        query_type=SearchType.GRAPH_COMPLETION,
        node_type=NodeSet,
        node_name=["principles_data"],
    )
    if not results:
        return "No relevant principles found."
    return str(results)


print("✅ Cognee tools defined: search_knowledge, search_principles")

In [ ]:
coding_agent = await provider.create_agent(
    name="CodingAssistant",
    instructions=(
        "You are an expert coding assistant with access to a knowledge graph "
        "containing developer profiles, Python best practices, and past conversations.\n\n"
        "WORKFLOW:\n"
        "1. Use search_knowledge() to find relevant information from the full knowledge graph.\n"
        "2. Use search_principles() when the question is specifically about Python best practices.\n"
        "3. Combine retrieved knowledge with your own expertise to give comprehensive answers.\n"
        "4. Reference the developer's known tech stack (FastAPI, asyncio, Pydantic) when relevant."
    ),
)

print("✅ CodingAssistant agent created")

## Memori Kerja dengan Sesi

`AgentSession` (dibuat melalui `agent.create_session()`) menyediakan memori kerja dalam satu sesi. Ejen boleh merujuk kembali kepada mesej sebelumnya sambil juga memeriksa graf pengetahuan jangka panjang Cognee.


In [ ]:
session = coding_agent.create_session()

response = await coding_agent.run(
    "How does my AsyncWebScraper implementation align with Python's design principles?",
    session=session,
)
print("🤖 Agent:", response)

In [ ]:
response = await coding_agent.run(
    "Based on what you just said, when should I pick dataclasses versus Pydantic for this work?",
    session=session,
)
print("🤖 Agent:", response)
print("\n💡 The agent combined working memory (previous answer) with Cognee's knowledge graph.")

## Sesi Baharu — Memori Jangka Panjang Kekal

Memulakan sesi baharu akan membersihkan memori kerja, tetapi graf pengetahuan Cognee masih tersedia. Ejen boleh mengambil semula pengetahuan jangka panjang yang sama dalam perbualan yang benar-benar baru.


In [ ]:
session_2 = coding_agent.create_session()

response = await coding_agent.run(
    "What logging guidance should I follow for incident reviews?",
    session=session_2,
)
print("🤖 Agent:", response)
print("\n💡 New session, but the agent still has access to the full Cognee knowledge graph.")

In [ ]:
response = await coding_agent.run(
    "How should variables be named according to Python best practices?",
    session=session_2,
)
print("🤖 Agent:", response)

## Ringkasan

Dalam buku nota ini anda membina pembantu pengaturcaraan yang menggabungkan **memori kerja MAF** (`agent.create_session()`) dengan **graf pengetahuan jangka panjang Cognee**.

### Apa yang Anda Telah Pelajari
1. **Pembinaan graf pengetahuan**: Cognee menelan teks tidak berstruktur dan membina graf + memori vektor.
2. **Pengayaan graf dengan memify**: Fakta yang diperoleh dan hubungan yang lebih kaya di atas graf sedia ada anda.
3. **Integrasi MAF + Cognee**: Fungsi `@tool` membolehkan ejen MAF bertanyakan graf Cognee secara semula jadi.
4. **Memori kerja + memori jangka panjang**: `AgentSession` (melalui `agent.create_session()`) menyediakan konteks sesi sementara Cognee menyediakan pengetahuan berterusan.
5. **Carian ditapis dengan NodeSets**: Sasarkan subset tertentu graf pengetahuan (contoh hanya prinsip).

### Intipati Utama
- **Cognee** menukar teks mentah menjadi memori berstruktur yang sedar hubungan — lebih berkuasa daripada stor vektor rata.
- **Fungsi `@tool`** menghubungkan ejen MAF dan sistem pengetahuan luaran dengan bersih.
- **`AgentSession`** (melalui `agent.create_session()`) mengekalkan konteks setiap perbualan secara berasingan daripada pengetahuan jangka panjang.
- Graf pengetahuan yang sama berkhidmat untuk pelbagai sesi dan ejen.

### Aplikasi Dunia Sebenar
- **Pembantu pemaju**: Semakan kod, analisis insiden, pembantu seni bina
- **Pembantu pelanggan**: Ejen sokongan ke atas dokumen produk, FAQ, dan nota CRM
- **Pembantu pakar dalaman**: Pembantu dasar, undang-undang, atau keselamatan yang membuat hujah berdasarkan garis panduan
- **Lapisan data bersatu**: Gabungkan data berstruktur dan tidak berstruktur dalam satu graf yang boleh dipersoalkan

### Langkah Seterusnya
- Bereksperimen dengan kesedaran temporal dalam Cognee
- Mentakrifkan ontologi OWL untuk kualiti graf khusus domain
- Menambah gelung maklum balas pengguna untuk memperbaiki pengambilan dari masa ke masa
- Skala ke sistem multi-ejen yang berkongsi lapisan memori Cognee yang sama


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Penafian**:  
Dokumen ini telah diterjemahkan menggunakan perkhidmatan terjemahan AI [Co-op Translator](https://github.com/Azure/co-op-translator). Walaupun kami berusaha untuk ketepatan, sila ambil perhatian bahawa terjemahan automatik mungkin mengandungi kesilapan atau ketidaktepatan. Dokumen asal dalam bahasa asalnya hendaklah dianggap sebagai sumber yang sahih. Untuk maklumat penting, terjemahan profesional oleh manusia adalah disarankan. Kami tidak bertanggungjawab atas sebarang salah faham atau salah tafsir yang timbul daripada penggunaan terjemahan ini.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
